<a href="https://colab.research.google.com/github/Moquiuti/LangChainePython/blob/main/chat_com_mem%C3%B3ria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 11.2 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

api_key = userdata.get("python-ai-integrada")
os.environ["GOOGLE_API_KEY"] = api_key

print("Chave carregada com sucesso!" if api_key else "Chave não encontrada.")

Chave carregada com sucesso!


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

In [4]:
modelo = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

Template de chat

In [5]:
prompt_chat = ChatPromptTemplate.from_messages([
    ("system", "Você é o Sr. Passeios, um assistente simpático e objetivo especializado em sugestões de viagem."),
    MessagesPlaceholder(variable_name="historico"),
    ("human", "{pergunta}")
])

Cadeia básica

In [6]:
cadeia_chat = prompt_chat | modelo | StrOutputParser()

Execução em lote

In [7]:
perguntas = [
    {"historico": [], "pergunta": "Me sugira uma cidade para viajar com foco em praias."},
    {"historico": [], "pergunta": "Me sugira uma cidade para uma viagem cultural no Brasil."},
    {"historico": [], "pergunta": "Me sugira um destino para viajar com crianças."}
]

respostas = cadeia_chat.batch(perguntas)

for i, resposta in enumerate(respostas, start=1):
    print(f"Resposta {i}:")
    print(resposta)
    print()

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 52.481985361s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-Fre

In [8]:
historicos_por_sessao = {}

Função estilo Singleton para recuperar histórico

In [9]:
def obter_historico(sessao_id: str):
    if sessao_id not in historicos_por_sessao:
        historicos_por_sessao[sessao_id] = []
    return historicos_por_sessao[sessao_id]

Função para conversar com memória

In [10]:
def conversar(sessao_id: str, pergunta: str):
    historico = obter_historico(sessao_id)

    resposta = cadeia_chat.invoke({
        "historico": historico,
        "pergunta": pergunta
    })

    historico.append(HumanMessage(content=pergunta))
    historico.append(AIMessage(content=resposta))

    return resposta

Testando memória na prática

In [11]:
sessao = "usuario_1"

print("Pergunta 1:")
print(conversar(sessao, "Quero viajar para um lugar com praias bonitas."))
print()

print("Pergunta 2:")
print(conversar(sessao, "Agora me indique restaurantes nesse destino."))
print()

print("Pergunta 3:")
print(conversar(sessao, "E uma dica cultural para complementar o passeio?"))
print()

Pergunta 1:


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Validando o histórico armazenado

In [12]:
for mensagem in historicos_por_sessao["usuario_1"]:
    print(type(mensagem).__name__, "->", mensagem.content)

Testando sessões independentes

In [13]:
print("Sessão A:")
print(conversar("sessao_a", "Quero um destino de serra para descansar."))
print()

print("Sessão B:")
print(conversar("sessao_b", "Quero um destino com praia e agito noturno."))
print()

print("Sessão A novamente:")
print(conversar("sessao_a", "Agora me sugira passeios nesse destino."))
print()

Sessão A:


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 40.848181482s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT